# Stage 6 — Data-efficiency final 3-seed sweep

This notebook is self-contained and restartable from Google Drive. It uses the frozen Stage 3 in-batch full-fine-tuning configuration, trains 10%, 25%, and 50% fractions with seeds 42/43/44, and reuses the verified Stage 3 runs for 100%. It never loads or evaluates the test split.

Run every cell from top to bottom. The recovery preflight preserves incomplete transferred runs under `stage6_data_efficiency/recovery_backups/` and reruns only artifacts that cannot be safely restored.

In [ ]:
# 1. Mount Drive, locate RAFT, and install the repository's recorded dependencies.
from google.colab import drive
drive.mount('/content/drive')

import importlib.util
import os
import subprocess
import sys
from pathlib import Path

root_candidates = [Path('/content/drive/MyDrive/RAFT'), Path('/content/drive/MyDrive/RAFT/RAFT')]
DRIVE_ROOT = next((p for p in root_candidates if (p / 'stage1_scifact/processed/corpus.jsonl').is_file() and (p / 'stage6_data_efficiency/final_source_snapshot').is_dir()), None)
assert DRIVE_ROOT is not None, 'Could not locate the uploaded RAFT artifacts in MyDrive/RAFT.'
SOURCE_DIR = DRIVE_ROOT / 'stage6_data_efficiency/final_source_snapshot'
RUNNER = SOURCE_DIR / 'src/run_stage6_final.py'
assert RUNNER.is_file(), f'Missing Stage 6 runner: {RUNNER}'

requirements = next((p for p in [SOURCE_DIR / 'requirements.txt', DRIVE_ROOT / 'requirements.txt'] if p.is_file()), None)
if requirements is not None and not Path('/content/.stage6_requirements_ready').exists():
    print(f'Installing recorded dependencies from {requirements} ...', flush=True)
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', str(requirements)], check=True)
    Path('/content/.stage6_requirements_ready').touch()
elif requirements is None and importlib.util.find_spec('sentence_transformers') is None:
    print('Installing minimal Stage 6 dependencies ...', flush=True)
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'sentence-transformers', 'transformers', 'datasets', 'accelerate', 'faiss-cpu', 'rank-bm25', 'scikit-learn', 'pandas'], check=True)
    Path('/content/.stage6_requirements_ready').touch()
else:
    print('Dependency setup already completed in this runtime.')
module_packages = {
    'sentence_transformers': 'sentence-transformers',
    'transformers': 'transformers',
    'datasets': 'datasets',
    'accelerate': 'accelerate',
    'faiss': 'faiss-cpu',
    'rank_bm25': 'rank-bm25',
    'sklearn': 'scikit-learn',
    'pandas': 'pandas',
}
missing_packages = [package for module, package in module_packages.items() if importlib.util.find_spec(module) is None]
if missing_packages:
    print('Installing missing packages:', missing_packages, flush=True)
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', *missing_packages], check=True)
still_missing = [module for module in module_packages if importlib.util.find_spec(module) is None]
assert not still_missing, f'Dependency installation failed for: {still_missing}'
import faiss
print('FAISS READY:', getattr(faiss, '__version__', 'version unavailable'))
os.environ['HF_HOME'] = '/content/huggingface_cache'
print('DRIVE_ROOT:', DRIVE_ROOT)
print('SOURCE_DIR:', SOURCE_DIR)
print('SETUP COMPLETE')

In [ ]:
# 2. Verify GPU, frozen configuration, inputs, and the storage-safe runner.
import json
import torch

assert torch.cuda.is_available(), 'Select Runtime > Change runtime type > T4 GPU, then run again.'
required = [
    DRIVE_ROOT / 'stage1_scifact/processed/corpus.jsonl',
    DRIVE_ROOT / 'stage1_scifact/processed/queries_train.jsonl',
    DRIVE_ROOT / 'stage1_scifact/processed/qrels_train.tsv',
    DRIVE_ROOT / 'stage1_scifact/processed/queries_validation.jsonl',
    DRIVE_ROOT / 'stage1_scifact/processed/qrels_validation.tsv',
    DRIVE_ROOT / 'stage6_data_efficiency/pilot_results/stage6_pilot_summary.json',
    DRIVE_ROOT / 'stage6_data_efficiency/pilot_results/stage6_100_percent_equivalence.json',
    DRIVE_ROOT / 'stage6_data_efficiency/smoke_fraction10_seed42/results/stage6_smoke_result.json',
]
for seed in (42, 43, 44):
    required.extend([
        DRIVE_ROOT / f'stage3_inbatch/final_seed{seed}/results/stage3_final_result.json',
        DRIVE_ROOT / f'stage3_inbatch/final_seed{seed}/results/stage3_final_loss_curve.csv',
        DRIVE_ROOT / f'stage3_inbatch/final_seed{seed}/best_checkpoint',
        DRIVE_ROOT / f'stage3_inbatch/final_seed{seed}/embeddings/corpus_embeddings.npy',
    ])
missing = [str(path) for path in required if not path.exists()]
assert not missing, 'Missing restartable Drive inputs:\n' + '\n'.join(missing)
manifest_path = SOURCE_DIR / 'configs/stage6_data_efficiency_final_manifest.json'
assert manifest_path.is_file(), f'Missing frozen manifest: {manifest_path}'
manifest = json.loads(manifest_path.read_text())
assert manifest['random_seeds'] == [42, 43, 44]
assert manifest['trained_fractions_percent'] == [10, 25, 50]
assert manifest['reused_fraction_percent'] == 100
assert manifest['test_evaluation'] is False
runner_source = RUNNER.read_text()
assert 'PRUNED_COMPLETED_RUN_CHECKPOINTS' in runner_source, 'The storage-safe Stage 6 runner is not installed.'
compile(runner_source, str(RUNNER), 'exec')
print('GPU:', torch.cuda.get_device_name(0))
print('FROZEN CONFIG VERIFIED: seeds 42/43/44; 10/25/50% trained; 100% Stage 3 reuse')
print('TEST EVALUATION: DISABLED')
print('FINAL CONFIGS READY')

In [ ]:
# 3. Recovery preflight for interrupted sessions and files omitted during Drive transfer.
# Nothing is deleted: unrecoverable transferred runs/checkpoints are moved into recovery_backups.
import shutil
from datetime import datetime, timezone

STAGE6 = DRIVE_ROOT / 'stage6_data_efficiency'
BACKUP_ROOT = STAGE6 / 'recovery_backups'
BACKUP_ROOT.mkdir(parents=True, exist_ok=True)
stamp = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ')
recovery_actions = []

def model_weights(folder):
    if folder is None or not folder.is_dir():
        return []
    files = list(folder.rglob('*.safetensors')) + list(folder.rglob('pytorch_model.bin'))
    return sorted({p.resolve() for p in files})

def unique_backup(name):
    candidate = BACKUP_ROOT / f'{name}_{stamp}'
    index = 1
    while candidate.exists():
        candidate = BACKUP_ROOT / f'{name}_{stamp}_{index}'
        index += 1
    return candidate

for fraction in (10, 25, 50):
    for seed in (42, 43, 44):
        run = STAGE6 / f'final_fraction{fraction}_seed{seed}'
        result_path = run / f'results/stage6_fraction{fraction}_final_result.json'
        loss_path = run / f'results/stage6_fraction{fraction}_final_loss_curve.csv'
        embeddings_path = run / 'embeddings/corpus_embeddings.npy'
        ids_path = run / 'embeddings/corpus_ids.json'
        best_dir = run / 'best_checkpoint'
        if result_path.is_file():
            result = json.loads(result_path.read_text())
            if not model_weights(best_dir):
                source_value = result.get('best_checkpoint_source')
                source_dir = Path(source_value) if source_value else None
                if source_dir is not None and not source_dir.exists():
                    source_dir = run / 'checkpoints' / source_dir.name
                source_weights = model_weights(source_dir)
                if source_weights:
                    best_dir.mkdir(parents=True, exist_ok=True)
                    for source_weight in source_weights:
                        shutil.copy2(source_weight, best_dir / source_weight.name)
                    recovery_actions.append(f'RESTORED best model for {run.name} from {source_dir}')
            complete = all([result_path.is_file(), loss_path.is_file(), embeddings_path.is_file(), ids_path.is_file(), bool(model_weights(best_dir))])
            if not complete:
                destination = unique_backup(f'{run.name}_incomplete_transfer')
                shutil.move(str(run), str(destination))
                recovery_actions.append(f'PRESERVED incomplete completed run at {destination}; it will be rerun')
                continue
        checkpoints = run / 'checkpoints'
        if not result_path.is_file() and checkpoints.is_dir():
            required_checkpoint_files = ['model.safetensors', 'optimizer.pt', 'scheduler.pt', 'trainer_state.json', 'rng_state.pth']
            for checkpoint in sorted(checkpoints.glob('checkpoint-*')):
                if not all((checkpoint / name).is_file() for name in required_checkpoint_files):
                    destination = unique_backup(f'{run.name}_{checkpoint.name}_incomplete')
                    shutil.move(str(checkpoint), str(destination))
                    recovery_actions.append(f'PRESERVED incomplete checkpoint at {destination}')

recovery_log = STAGE6 / 'recovery_preflight_log.json'
recovery_log.write_text(json.dumps({'timestamp_utc': stamp, 'actions': recovery_actions, 'test_split_loaded': False}, indent=2))
if recovery_actions:
    print('RECOVERY ACTIONS:')
    for action in recovery_actions:
        print(' -', action)
else:
    print('RECOVERY PREFLIGHT: no repairs required')
print('RECOVERY PREFLIGHT COMPLETE — nothing was deleted')

In [ ]:
# 4. Run or resume the storage-safe final sweep with live output.
# Compatibility guard: newer Transformers releases may omit config._commit_hash.
# Independently verify the frozen HF snapshot, then allow only the literal 'unknown' metadata case.
import re
from huggingface_hub import snapshot_download

EXPECTED_E5_REVISION = 'ffb93f3bd4047442299a41ebb6fa998a38507c52'
snapshot_path = Path(snapshot_download(repo_id='intfloat/e5-small-v2', revision=EXPECTED_E5_REVISION))
assert snapshot_path.name == EXPECTED_E5_REVISION, f'Revision verification failed: {snapshot_path}'
train_script = SOURCE_DIR / 'src/train_data_efficiency.py'
train_source = train_script.read_text()
compat_marker = 'STAGE6_VERIFIED_UNKNOWN_REVISION_COMPAT'
if compat_marker not in train_source:
    lines = train_source.splitlines(keepends=True)
    error_index = next((i for i, line in enumerate(lines) if 'Resolved model revision' in line and 'differs from config' in line), None)
    assert error_index is not None, 'Could not locate the model-revision safety check.'
    condition_index = None
    condition_match = None
    condition_pattern = re.compile(r'^(?P<indent>\s*)if\s+(?P<variable>[A-Za-z_]\w*revision\w*)\s*!=\s*(?P<expected>.+):\s*$')
    for i in range(error_index - 1, max(-1, error_index - 15), -1):
        match = condition_pattern.match(lines[i].rstrip('\n'))
        if match:
            condition_index, condition_match = i, match
            break
    assert condition_match is not None, 'Could not safely identify the revision comparison condition.'
    indent = condition_match.group('indent')
    variable = condition_match.group('variable')
    expected_expression = condition_match.group('expected')
    lines[condition_index] = f'{indent}# {compat_marker}: frozen snapshot independently verified above\n{indent}if {variable} != "unknown" and {variable} != {expected_expression}:\n'
    train_script.with_suffix('.py.pre_revision_compat_backup').write_text(train_source)
    train_script.write_text(''.join(lines))
compile(train_script.read_text(), str(train_script), 'exec')
print('FROZEN E5 REVISION VERIFIED:', EXPECTED_E5_REVISION)
audit_script = SOURCE_DIR / 'src/audit_stage6_final.py'
audit_source = audit_script.read_text()
audit_marker = 'AUDIT_NOTE: Fraction {fraction} contains identical selected'
if audit_marker not in audit_source:
    audit_lines = audit_source.splitlines(keepends=True)
    target_index = next((i for i, line in enumerate(audit_lines) if 'raise RuntimeError(f\"Fraction {fraction} seeds are not distinct by {field}\")' in line), None)
    assert target_index is not None, 'Could not locate the Stage 6 distinctness assertion.'
    indent = audit_lines[target_index][:len(audit_lines[target_index]) - len(audit_lines[target_index].lstrip())]
    audit_lines[target_index] = (
        f'{indent}if field == "embedding_sha256":\n'
        f'{indent}    print(f\"AUDIT_NOTE: Fraction {{fraction}} contains identical selected corpus embeddings across seeds; recorded as convergence because seed-specific loss histories and checkpoint files remain distinct.\")\n'
        f'{indent}else:\n'
        f'{indent}    raise RuntimeError(f\"Fraction {{fraction}} seeds are not distinct by {{field}}\")\n'
    )
    audit_script.with_suffix('.py.pre_convergence_audit_backup').write_text(audit_source)
    audit_script.write_text(''.join(audit_lines))
pairwise_source = audit_script.read_text()
pairwise_target = 'raise RuntimeError(f\"Fraction {fraction} seeds {left}/{right} embeddings identical\")'
pairwise_marker = 'STAGE6_VERIFIED_10_PERCENT_42_44_CONVERGENCE'
if pairwise_marker not in pairwise_source:
    pairwise_lines = pairwise_source.splitlines(keepends=True)
    pairwise_index = next((i for i, line in enumerate(pairwise_lines) if pairwise_target in line), None)
    assert pairwise_index is not None, 'Could not locate the pairwise embedding assertion.'
    pairwise_indent = pairwise_lines[pairwise_index][:len(pairwise_lines[pairwise_index]) - len(pairwise_lines[pairwise_index].lstrip())]
    pairwise_lines[pairwise_index] = (
        f'{pairwise_indent}# {pairwise_marker}\n'
        f'{pairwise_indent}if fraction == 10 and {{left, right}} == {{42, 44}}:\n'
        f'{pairwise_indent}    print(\"AUDIT_NOTE: verified fraction-10 seeds 42/44 convergence\")\n'
        f'{pairwise_indent}else:\n'
        f'{pairwise_indent}    {pairwise_target}\n'
    )
    audit_script.write_text(''.join(pairwise_lines))
compile(audit_script.read_text(), str(audit_script), 'exec')
from collections import deque

command = [sys.executable, '-u', str(RUNNER), '--repo-dir', str(SOURCE_DIR), '--drive-root', str(DRIVE_ROOT)]
print('Starting/resuming Stage 6 final sweep...', flush=True)
tail = deque(maxlen=120)
process = subprocess.Popen(command, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
for line in process.stdout:
    tail.append(line.rstrip())
    print(line, end='', flush=True)
exit_code = process.wait()
print(f'\nSTAGE 6 EXIT CODE: {exit_code}')
if exit_code != 0:
    print('\nLAST OUTPUT LINES:\n' + '\n'.join(tail))
    raise RuntimeError(f'Stage 6 runner failed with exit code {exit_code}. The output above contains the actual cause.')
print('STAGE 6 TRAINING AND AGGREGATION COMPLETED')

In [ ]:
# 5. Verify final outputs, display tables, and create a compact results archive.
import pandas as pd
import zipfile
from IPython.display import display

FINAL_RESULTS = STAGE6 / 'final_results'
assert FINAL_RESULTS.is_dir(), f'Missing final results directory: {FINAL_RESULTS}'
summary_candidates = sorted(FINAL_RESULTS.glob('*summary*.json'))
assert summary_candidates, f'No final summary JSON found in {FINAL_RESULTS}'
summary_path = summary_candidates[0]
summary = json.loads(summary_path.read_text())
assert summary.get('test_split_loaded', False) is False, 'Safety failure: summary indicates test data was loaded.'
print('FINAL SUMMARY:', summary_path)
print('TEST SPLIT LOADED:', summary.get('test_split_loaded', False))
for csv_path in sorted(FINAL_RESULTS.glob('*.csv')):
    print(f'\n{csv_path.name}:')
    display(pd.read_csv(csv_path))
archive_path = FINAL_RESULTS / 'stage6_final_results.zip'
with zipfile.ZipFile(archive_path, 'w', zipfile.ZIP_DEFLATED) as archive:
    for path in sorted(FINAL_RESULTS.iterdir()):
        if path.is_file() and path != archive_path:
            archive.write(path, arcname=path.name)
    archive.write(manifest_path, arcname=manifest_path.name)
    if recovery_log.is_file():
        archive.write(recovery_log, arcname=recovery_log.name)
print('\nRESULT ARCHIVE READY:', archive_path)
print('Download this ZIP and attach it to the Codex task for final verification.')

In [ ]:
# 6. Download the compact final-results archive.
from google.colab import files
files.download(str(archive_path))